# CSA + prompt → SCST + CE + fixed rerank — PROMPT

Một notebook/một tài khoản, **GPU T4 ×2**, Internet ON, Save Version → Save & Run All. Nạp **selected PROMPT từ ducanh3/notebook7bc83c4c76/58**, không train từ đầu, không reset adapter.

Gắn4Dataset: `csa-full-roi-karpathy-v134`, `csa-prompt-full-karpathy-v175`, `vuthetam/mscoco-2014` (**cả JPEG gốc và dataset_coco.json**), `objectdetectionecache`.
Không cần bank130 riêng: bank nằm trong checkpoint. Dataset175 phải được chia sẻ quyền cho tài khoản chạy.

Nếu chưa lưu checkpoint thành Dataset, bật Secret tên `codex` (API token của **chính tài khoản đang chạy**). Notebook chỉ tải best.bin đúng version của tài khoản đó. Nếu gắn checkpoint sẵn, điền WEIGHTS_ROOT, không cần API.

Giữ nguyên kiến trúc. CONTROL tắt adapter; PROMPT dùng adapter đã học. Học **1 lượt toàn bộ113287ảnh TRAIN =14161update/global8**, cuối4/3; freshAdam5e-6, CE+.05SCST, không thưởng đối tượng.
Trước/sau học: fullVALID5000, beam5/max20; cùng fixed rerank .3decoder+.4CLIP+.3object. KhôngBLIP/TEST/tuning/oracle. C10/P9 có số epoch XE ban đầu khác nhau; không gọi là bằng nhau tổng ngân sách.

Giới hạn4giờ theo đồng hồ notebook, không phải cam kết thời gian hay quota tính phí. Lưu checkpoint mỗi500update và khi dừng; giữ cả output nếu incomplete. Hai tài khoản có quota10/15giờ theo báo cáo mới; không tự tăngcap/epoch.
Tác giả chỉ kiểm NumPy/metadata/AST/mock, không chạy model/DDP/notebook. Gửi ZIPreview và giữ checkpoint.bin/nguồn XE/candidates.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
ARM='PROMPT'
KERNEL_VERSION='ducanh3/notebook7bc83c4c76/58' # This account's own output only; never replace with latest.

SECRET_NAME='codex'
WEIGHTS_ROOT='' # Optional local mount containing the exact selected best.bin; blank downloads own version.
RESUME_ROOT='' # Only a complete preserved output from THIS arm/source/recipe. Blank starts the approved continuation.
WALL_HOURS=4 # Parent STOPcap, not a promised runtime or billed GPU quota.
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
COCO_ROOT=Path('/kaggle/input/datasets/vuthetam/mscoco-2014')
COCO_JSON=COCO_ROOT/'dataset_coco.json'
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/csa_prompt_scst_'+ARM+'_full1_v1')
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
for p in (FULL_ROOT,PROMPT_ROOT,COCO_ROOT,COCO_JSON,DETECTIONS):
    if not p.exists():raise FileNotFoundError('Attach inputs BEFORE GPU; correct this path: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve prior output. Use a fresh session/output path; no automatic restart.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<10*1024**3:raise RuntimeError('Need10GiB free for ROI stage, weights and atomic checkpoints')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-prompt-scst-'+ARM.lower()+'-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
REVISION='3d39c94e068196528c1567c548386e007032faf7'
MAPPING={'experiments/csa_paper/RERANK_TEST_INPUTS.json': 'RERANK_TEST_INPUTS.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'experiments/csa_paper/PROMPT_SCST_PROTOCOL.md': 'PROMPT_SCST_PROTOCOL.md', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/make_prompt_scst_notebooks.py': 'make_prompt_scst_notebooks.py', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'experiments/csa_paper/test_prompt_adapter_model.py': 'test_prompt_adapter_model.py', 'experiments/csa_paper/prompt_adapter_model.py': 'prompt_adapter_model.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/test_scst_full_model.py': 'test_scst_full_model.py', 'experiments/csa_paper/scst_pilot_model.py': 'scst_pilot_model.py', 'experiments/csa_paper/test_scst_pilot_model.py': 'test_scst_pilot_model.py', 'experiments/csa_paper/test_prompt_scst_data.py': 'test_prompt_scst_data.py', 'experiments/csa_paper/prompt_scst_rerank.py': 'prompt_scst_rerank.py', 'experiments/csa_paper/run_scst_pilot.py': 'run_scst_pilot.py', 'experiments/csa_paper/scst_full_data.py': 'scst_full_data.py', 'experiments/csa_paper/scst_pilot_data.py': 'scst_pilot_data.py', 'captioning/object_reward.py': 'captioning/object_reward.py', 'captioning/object_consistency.py': 'captioning/object_consistency.py', 'experiments/csa_paper/full_xe_data.py': 'full_xe_data.py', 'experiments/csa_paper/sharing_data.py': 'sharing_data.py', 'experiments/csa_paper/full_cache_data.py': 'full_cache_data.py', 'experiments/csa_paper/package_full_roi_dataset.py': 'package_full_roi_dataset.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'captioning/scst.py': 'captioning/scst.py', 'experiments/csa_paper/rerank_data.py': 'rerank_data.py', 'captioning/multiscore_reranking.py': 'captioning/multiscore_reranking.py', 'experiments/csa_paper/candidate_data.py': 'candidate_data.py', 'experiments/csa_paper/prompt_scst_data.py': 'prompt_scst_data.py', 'experiments/csa_paper/prompt_scratch_data.py': 'prompt_scratch_data.py', 'experiments/csa_paper/full_prompt_cache_data.py': 'full_prompt_cache_data.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'captioning/upt_prompts.py': 'captioning/upt_prompts.py', 'captioning/upt_action_text.py': 'captioning/upt_action_text.py', 'captioning/selected_relation_prompts.py': 'captioning/selected_relation_prompts.py', 'captioning/supervised_relations.py': 'captioning/supervised_relations.py', 'experiments/csa_paper/run_prompt_scst.py': 'run_prompt_scst.py', 'experiments/csa_paper/prompt_pilot_model.py': 'prompt_pilot_model.py', 'experiments/csa_paper/prompt_scratch_model.py': 'prompt_scratch_model.py', 'experiments/csa_paper/full_xe_decode.py': 'full_xe_decode.py', 'captioning/__init__.py': 'captioning/__init__.py'}
HASHES={'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'experiments/csa_paper/PROMPT_SCST_PROTOCOL.md': '208be3aa8f45b3df2c55e9b4e5e25b0e9f1d0cb63b1069b0d1e5a016c46015ac', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/make_prompt_scst_notebooks.py': 'c53390a9b866fa8768642212c35293dd9cc7a1afda653824dbc3a06e2cc7eb64', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'experiments/csa_paper/test_prompt_adapter_model.py': '152d7419238d9484c5b16e84fd987f3e9f68a57649f17019e5532860e144554a', 'experiments/csa_paper/prompt_adapter_model.py': '83e5fdb2662940ded06e56df3a0fdf512dc1046b2352e563b59f22a6117e245c', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/test_scst_full_model.py': 'b3eb3af4872953f81d300f888e41c67478d284f417c67a43e00d89c9fe7e9814', 'experiments/csa_paper/scst_pilot_model.py': '0b5e2d384ffc7195c82888f3f7904eac145ba7291ebed27dc8342ed8e1ac9576', 'experiments/csa_paper/test_scst_pilot_model.py': 'd1192a844ad111f3423d39f6cdece1daca88417b3255bd44fd1f096bc9588ed5', 'experiments/csa_paper/test_prompt_scst_data.py': '09e1e00921935cd44670a5ac684a5f314c5223e44f94ea5eddd2ab8b40c654e9', 'experiments/csa_paper/prompt_scst_rerank.py': '09a69fe3d0ba1a8b339ddbf2b5ee3bb17ceb1547f79832d3f278cb3917141371', 'experiments/csa_paper/run_scst_pilot.py': '9a640766bd063d2e5d5854dab9177168ae6528bf0fcd43467ba7e5b38618285e', 'experiments/csa_paper/scst_full_data.py': 'f1d88bd3b5bdf5f7e873b128a9624ee645db7b9c075824782572c7e87a87240b', 'experiments/csa_paper/scst_pilot_data.py': '119324b3933e07a07ff5aa60a4cb4272ebfdd587d0451e56f338296954a2743b', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'experiments/csa_paper/full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'experiments/csa_paper/sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'experiments/csa_paper/full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'experiments/csa_paper/package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'experiments/csa_paper/rerank_data.py': '9621a0c3cabc27fcddb7fc891e31bd704c0bf5028721409a9f691d003c397dde', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'experiments/csa_paper/candidate_data.py': 'c9f8cf09d2ab72372eb1fffdc4b56bb097b12e8dd46aac68c60d60b65cf04257', 'experiments/csa_paper/prompt_scst_data.py': '68e6e77a6b2f0d095ce86eb0fcf36b8829aa877ef6e2cfd2623a6fc2d1b1708f', 'experiments/csa_paper/prompt_scratch_data.py': '025496a549e12a49104992e1c48f09968ebf4dcb520d649c818df6b39c0a6959', 'experiments/csa_paper/full_prompt_cache_data.py': 'efb21689e1983e8ab22f9d63068cf262f58344498d412056580c8af54c481f16', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'captioning/upt_prompts.py': '25bb24346d2b2c649ee7091c72d5def060ddf2c290524a35ef3d301d274fb65c', 'captioning/upt_action_text.py': 'ca0d8f1dc75be3ccfd589fcddc962e87a43222751a4d1beb68098cfa213d526c', 'captioning/selected_relation_prompts.py': 'de13dfbd80e4d4b8c08cd3e40407d5134cb1a5108c58f3eebfd4b74e237179d2', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984', 'experiments/csa_paper/run_prompt_scst.py': '72ca4a21f9292c3792c84b7f608c3eac6cb465ce3ec2b9f20ae394eecb5c197b', 'experiments/csa_paper/prompt_pilot_model.py': 'e47c869f7ef74af6ab734b2ac5de0973a53aae0234c2f56141a432de1d7a430c', 'experiments/csa_paper/prompt_scratch_model.py': '31db302359b2e6f850e4b1714a93c46555bf512c9b8afd86bbca6fcc8b6ffec9', 'experiments/csa_paper/full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633'}

for original,destination in MAPPING.items():
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline during source download')
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+original,timeout=min(90,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=HASHES[original]:raise RuntimeError('Changed pinned source:'+original)
    target=SOURCE/destination;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
os.environ['OMP_NUM_THREADS']='1';os.environ['PYTHONPATH']=str(SOURCE)
CHAIR=SOURCE/'chair';CONFIG=SOURCE/'config.json';DEPS=SOURCE/'hf-deps'
cfg=dict(arm=ARM,updates=14161,full_root=str(FULL_ROOT),prompt_root=str(PROMPT_ROOT),checkpoint_root=WEIGHTS_ROOT,
    dataset_json=str(COCO_JSON),coco_root=str(COCO_ROOT),detections=str(DETECTIONS),output=str(OUTPUT),stage=str(SOURCE/'stage'),
    chair=str(CHAIR),model_cache=str(SOURCE/'hf-cache'),worker_seconds=10000,rerank_seconds=1200,resume_root=RESUME_ROOT)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
    (OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')
write_config()


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent deadline exceeded')
    timeout=min(timeout,max(.01,PARENT_DEADLINE-time.monotonic()))
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Prompt SCST active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Parent/phase deadline exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_scst_data.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],timeout=60)
    if not WEIGHTS_ROOT:
        run([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],timeout=300)
        # Local mock checks exact version routing before the authenticated download.
        route="from types import SimpleNamespace;from unittest.mock import patch;from kaggle.api.kaggle_api_extended import KaggleApi;import tempfile;api=KaggleApi();owner,slug,version="+repr(KERNEL_VERSION)+".split('/');ctx=tempfile.TemporaryDirectory();d=ctx.__enter__();factory=patch.object(api,'build_kaggle_client');f=factory.__enter__();ep=f.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output;ep.return_value=SimpleNamespace(files=[],next_page_token=None,log=None);api.kernels_output("+repr(KERNEL_VERSION)+",path=d);r=ep.call_args.args[0];assert (r.user_name,r.kernel_slug,r.version_label)==(owner,slug,'v'+version);factory.__exit__(None,None,None);ctx.__exit__(None,None,None);print('Exact version route mock PASS')"
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret(SECRET_NAME).strip()
        if not token:raise RuntimeError('Enable configured Kaggle API Secret for THIS account')
        previous={k:os.environ.pop(k,None) for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY')}
        try:
            os.environ['KAGGLE_API_TOKEN']=token
            run([sys.executable,'-c',route],timeout=60)
            weight_dir=OUTPUT/'source_xe';weight_dir.mkdir()
            run([sys.executable,'-c','from kaggle.cli import main;main()','kernels','output',KERNEL_VERSION,
                 '-p',weight_dir,'--file-pattern',r'(^|/)best\.bin$'],timeout=900)
        finally:
            for k in previous:os.environ.pop(k,None)
            for k,v in previous.items():
                if v is not None:os.environ[k]=v
            del token
        cfg['checkpoint_root']=str(weight_dir)
    sys.path.insert(0,str(SOURCE))
    from prompt_scst_data import source_checkpoint
    print('Exact selected checkpoint:',source_checkpoint(cfg['checkpoint_root'],ARM),flush=True);write_config()
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    # Fail inference dependencies/assets before expensive training; isolate HF packages from host apps.
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'transformers==4.57.1','tokenizers==0.22.1',
         'huggingface-hub==0.36.2','safetensors==0.6.2'],timeout=300)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"from importlib.metadata import version;from transformers import CLIPModel,CLIPProcessor;assert version('transformers')=='4.57.1';assert version('huggingface-hub')=='0.36.2';print('Isolated HF imports PASS')"],timeout=60)
    run([sys.executable,'-u',SOURCE/'prompt_scst_rerank.py','prefetch','--config',CONFIG],timeout=1200)
    run([sys.executable,'-u',SOURCE/'run_prompt_scst.py','prepare','--config',CONFIG],timeout=1800)
    run([sys.executable,'-u',SOURCE/'prompt_scst_rerank.py','prepare','--config',CONFIG],timeout=600)
    run([sys.executable,'-u',SOURCE/'run_prompt_scst.py','metric_preflight','--config',CONFIG],timeout=240)
    run([sys.executable,'-u',SOURCE/'test_prompt_adapter_model.py'],timeout=180)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'test_scst_full_model.py'],timeout=240)
    run([*launch,SOURCE/'run_prompt_scst.py','preflight','--config',CONFIG],timeout=600)
    cfg['worker_seconds']=PARENT_DEADLINE-time.monotonic()-1800
    if cfg['worker_seconds']<2400:raise TimeoutError('Insufficient parent budget after preflight; preserve inputs/logs')
    write_config()
    run([*launch,SOURCE/'run_prompt_scst.py','worker','--config',CONFIG],timeout=cfg['worker_seconds']+30)
    run([sys.executable,'-u',SOURCE/'run_prompt_scst.py','score','--config',CONFIG],timeout=600)
    cfg['rerank_seconds']=PARENT_DEADLINE-time.monotonic()-600
    if cfg['rerank_seconds']<180:raise TimeoutError('Training complete, insufficient rerank reserve; preserve5beams, do not retrain')
    write_config()
    run([*launch,SOURCE/'prompt_scst_rerank.py','worker','--config',CONFIG],timeout=cfg['rerank_seconds']+30)
    run([sys.executable,'-u',SOURCE/'prompt_scst_rerank.py','score','--config',CONFIG],timeout=600)
    complete=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))['status']=='PROMPT_SCST_RERANK_VALID_COMPLETE'
    if not complete:raise RuntimeError('Incomplete full result')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,updates_requested=14161,test_used=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt') and p.name!='training_references.json':z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',REVIEW,'\nKEEP ENTIRE OUTPUT:',OUTPUT,'\nCHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)
    print('COMPLETE — ORIGINAL/END, rank0/fixed rerank, full VALID5000.' if complete else 'INCOMPLETE — preserve output. No automatic restart or partial-model ranking.',flush=True)
